# DN-SC: reproduce the GPT-2 study on a GPU and compare every number

This notebook re-runs the study behind `results/gpt2` from scratch on a Colab GPU (Runtime → Change runtime type → **T4 GPU**) and compares the outcome with the published numbers.

1. **Setup** — clone the repository and install it.
2. **Self-test** — the test suite, then the complete protocol on a tiny model *on the GPU*. If this passes, the code runs on this machine.
3. **Reproduction** — the chosen profile with pretrained GPT-2. Finished runs are skipped, so the cell can be re-run after an interruption.
4. **Verdict** — `COMPARISON.md`: published and reproduced value of every number, a tolerance check and a check of each conclusion.

A CPU and a GPU do not produce bit-identical floating-point results, so agreement is judged against a tolerance (3 % on the headline numbers) and against the conclusions. Nothing in this notebook edits a result by hand.

In [ ]:
# ---- settings -------------------------------------------------------------------------
REPO    = "https://github.com/abraheemrashid99-sys/Wiley_paper.git"
BRANCH  = "main"
PROFILE = "published"   # "published": the 37 published runs | "core": six seeds | "paper": everything
PERSIST_ON_DRIVE = False  # True: keep runs on Google Drive, so a disconnected session can be resumed

In [ ]:
import os, subprocess, sys, pathlib

def sh(command, check=True):
    print("$", command, flush=True)
    code = subprocess.run(command, shell=True).returncode
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")
    return code

if not pathlib.Path("scripts/reproduce.py").exists():      # not yet inside the repository
    if not pathlib.Path("Wiley_paper").exists():
        sh(f"git clone --depth 1 --branch {BRANCH} {REPO} Wiley_paper")
    os.chdir("Wiley_paper")
print("working directory:", os.getcwd())
sh(f'{sys.executable} -m pip install -q -e ".[dev,gpt2]"')

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU: this will be slow")

In [ ]:
# Optional: keep the run directory on Google Drive (Colab only)
RUNS = "runs"
if PERSIST_ON_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUNS = "/content/drive/MyDrive/dnsc_runs"
        os.makedirs(RUNS, exist_ok=True)
    except Exception as error:
        print("Drive not available, runs stay in this session:", error)
OUT, EXPORT = f"{RUNS}/gpt2_{PROFILE}", f"results_repro/gpt2_{PROFILE}"
print("runs:", OUT, "| exported results:", EXPORT)

## 1. Self-test

The unit and integration tests (CPU, tiny model), then the whole reproduction protocol on a tiny model on the selected device. The numbers of the tiny model mean nothing; what matters is that every step runs here.

In [ ]:
sh(f"{sys.executable} -m pytest -q -x")

In [ ]:
# shortened streams: the self-test checks that every step runs on this device, nothing else
SHORT = "stream.tau_slow=14 stream.tau_med=7 stream.temporal_steps=28 pretrain.steps=100 pretrain.eval_every=50 eval.n_test=16"
sh(f"{sys.executable} scripts/reproduce.py --config configs/smoke.yaml --out {RUNS}/selftest --export results_repro/selftest --profile published device={DEVICE} {SHORT}")

## 2. Reproduction with pretrained GPT-2

The first cell builds GPT-2 (124M) from files hosted on GitHub (checksum verified). The second runs the study; it prints one line per finished step and can simply be run again if the session was interrupted.

In [ ]:
if not pathlib.Path("models/gpt2/config.json").exists():
    sh(f"{sys.executable} scripts/fetch_gpt2.py --out models/gpt2")

In [ ]:
sh(f"{sys.executable} scripts/reproduce.py --config configs/gpt2.yaml --out {OUT} --export {EXPORT} --reference results/gpt2 --profile {PROFILE} device={DEVICE}")

## 3. Verdict

The comparison report. `REPRODUCED` means every headline number is within the tolerance and every conclusion holds; otherwise the report lists exactly what differs.

In [ ]:
from IPython.display import Markdown, display
report = pathlib.Path(EXPORT) / "COMPARISON.md"
display(Markdown(report.read_text(encoding="utf-8")))

In [ ]:
# The reproduced tables, figures and result files as one archive
import shutil
archive = shutil.make_archive(f"dnsc_{PROFILE}_reproduction", "zip", EXPORT)
print("archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass  # not on Colab: the archive is in the working directory